# JEV Trading benchmark kernel

This notebook is a thin compute wrapper around the repository's existing experiment scripts. The private Kaggle dataset supplies the current source snapshot and a pre-OOS parquet file. It records timings and packages results as `results.zip`.

Modes are selected by `scripts/build_kaggle_kernel.py --mode ...`: `smoke`, `phase0`, `phase1`, `phase2`, `ablation`, or `all`. The notebook does not change split, cost, or OOS rules.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
KAGGLE_INPUT = Path('/kaggle/input')
if KAGGLE_INPUT.is_dir():
    config_files = sorted(KAGGLE_INPUT.glob('*/run_config.json'))
    if len(config_files) != 1:
        raise FileNotFoundError(f'Expected one benchmark dataset under {KAGGLE_INPUT}, found {config_files}')
    INPUT_DIR = config_files[0].parent
else:
    INPUT_DIR = ROOT
CONFIG = json.loads((INPUT_DIR / 'run_config.json').read_text())
EXPECTED_MODE = '__JEV_MODE_FROM_BUILD__'
EXPECTED_SOURCE_SHA256 = '__JEV_SOURCE_HASH_FROM_BUILD__'
EXPECTED_DATA_SHA256 = '__JEV_DATA_HASH_FROM_BUILD__'
MODE = os.environ.get('JEV_RUN_MODE', CONFIG['mode'])
if EXPECTED_MODE != '__JEV_UNBUILT_NOTEBOOK__':
    if CONFIG.get('mode') != EXPECTED_MODE or MODE != EXPECTED_MODE:
        raise RuntimeError(f'Kaggle dataset mode mismatch: expected {EXPECTED_MODE}, got {CONFIG.get("mode")}/{MODE}')
    if CONFIG.get('source_archive_sha256') != EXPECTED_SOURCE_SHA256 or CONFIG.get('data_sha256') != EXPECTED_DATA_SHA256:
        raise RuntimeError('Kaggle dataset source or data hash does not match this kernel build')
THREADS = str(CONFIG.get('threads', 2))
for _name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'POLARS_MAX_THREADS'):
    os.environ[_name] = THREADS

REPO = ROOT / 'repo'
RESULTS = ROOT / 'results'
if REPO.exists():
    shutil.rmtree(REPO)
if RESULTS.exists():
    shutil.rmtree(RESULTS)
REPO.mkdir(parents=True)
RESULTS.mkdir(parents=True)
source_dir = INPUT_DIR / 'source'
if source_dir.is_dir():
    shutil.copytree(source_dir, REPO, dirs_exist_ok=True)
else:
    with tarfile.open(INPUT_DIR / 'source.tar.gz', 'r:gz') as archive:
        archive.extractall(REPO)
os.chdir(REPO)

def run_step(name, command):
    print(f'[{name}] {" ".join(map(str, command))}', flush=True)
    started = time.perf_counter()
    result = subprocess.run(list(map(str, command)), cwd=REPO)
    elapsed = time.perf_counter() - started
    timings[name] = {'seconds': round(elapsed, 3), 'returncode': result.returncode}
    (RESULTS / 'timings.json').write_text(json.dumps(timings, indent=2, sort_keys=True))
    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print(f'[{name}] completed in {elapsed:.1f}s', flush=True)

timings = {}
run_step('install', [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'])
run_step('git-init', ['git', 'init', '-q'])
run_step('git-config-email', ['git', 'config', 'user.email', 'kaggle@localhost'])
run_step('git-config-name', ['git', 'config', 'user.name', 'Kaggle source snapshot'])
run_step('git-add', ['git', 'add', '-A'])
run_step('git-commit', ['git', 'commit', '-qm', 'Kaggle source snapshot'])
kernel_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
(RESULTS / 'run_config.json').write_text(json.dumps({**CONFIG, 'mode_used': MODE, 'kernel_commit': kernel_commit}, indent=2, sort_keys=True))
source_manifest = json.loads((REPO / 'KAGGLE_SOURCE_MANIFEST.json').read_text())
(RESULTS / 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2, sort_keys=True))
DATA = INPUT_DIR / CONFIG['data_file']
if not DATA.is_file():
    raise FileNotFoundError(f'Bundled data file is missing: {DATA}')

import polars as pl
timestamps = pl.read_parquet(DATA, columns=['timestamp'])['timestamp']
cutoff = int(datetime(2025, 1, 1, tzinfo=timezone.utc).timestamp() * 1000)
if timestamps.is_empty() or int(timestamps.max()) >= cutoff:
    raise ValueError('The bundled dataset is empty or contains frozen OOS rows')
print(f'mode={MODE}; pre-OOS rows={timestamps.len()}; max_timestamp={int(timestamps.max())}', flush=True)
del timestamps

run_step('pytest', [sys.executable, '-m', 'pytest', '-q'])


In [ ]:
phase0_command = [
    sys.executable, 'scripts/reproduce_phase0.py',
    '--source', str(DATA), '--out', str(RESULTS / 'phase0'),
]
phase1_command = [
    sys.executable, 'scripts/run_phase1_economic.py',
    '--bars', str(DATA), '--out', str(RESULTS / 'phase1'),
]
phase2_command = [
    sys.executable, 'scripts/run_phase2_experiments.py',
    '--bars', str(DATA), '--out', str(RESULTS / 'phase2'),
    '--trees', str(CONFIG['trees']), '--leaves', str(CONFIG['leaves']),
]
if CONFIG.get('skip_derived'):
    phase2_command.append('--skip-derived')

if MODE in {'phase0', 'ablation', 'all'}:
    run_step('phase0', phase0_command)
if MODE in {'phase1', 'all'}:
    run_step('phase1', phase1_command)
if MODE in {'phase2', 'all'}:
    run_step('phase2', phase2_command)
if MODE in {'ablation', 'all'}:
    ablation_command = [
        sys.executable, 'scripts/run_ablation.py',
        '--bars', str(DATA), '--models', str(RESULTS / 'phase0' / 'models'),
        '--events', str(RESULTS / 'ablation' / 'events'),
        '--start', '2024-01-01', '--end', '2025-01-01',
        '--arms', 'random,threshold,policy,full,nojev',
        '--fee-mult', '1.0,2.0,3.0', '--no-logs',
    ]
    run_step('ablation', ablation_command)
if MODE == 'smoke':
    print('Smoke mode completed: environment, data boundary, and test suite are valid.', flush=True)
elif MODE not in {'phase0', 'phase1', 'phase2', 'ablation', 'all'}:
    raise ValueError(f'Unknown mode: {MODE}')


In [ ]:
status = {
    'status': 'passed',
    'mode': MODE,
    'finished_utc': datetime.now(timezone.utc).isoformat(),
    'kernel_commit': kernel_commit,
    'base_commit': CONFIG.get('base_commit'),
}
(RESULTS / 'run_status.json').write_text(json.dumps(status, indent=2, sort_keys=True))
archive_path = ROOT / 'results.zip'
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS).as_posix())
print(f'Wrote {archive_path} ({archive_path.stat().st_size} bytes)', flush=True)
